In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df_raw = pd.read_csv('data/results.csv')
df_raw.head()

/var/folders/7q/hg2_dwc17jng9c7169fzwrb80000gn/T/ipykernel_6074/1162668877.py:1: DtypeWarning: Columns (56: JobSatPoints_15_TEXT, 74: DatabaseHaveEntry, 92: DevEnvHaveEntry, 97: SOTagsHaveEntry, 98: SOTagsWant Entry, 105: OfficeStackWantEntry, 109: CommPlatformHaveEntr, 110: CommPlatformWantEntr, 132: SO_Actions_15_TEXT, 162: AIAgentOrchestration, 165: AIAgentObsWrite) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('data/results.csv')


,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",None of the above,10.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,"Understand how things actually work, problem s...",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,4.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to...,NaN,36197.0,6.0
4,5,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...","Caring for dependents (children, elderly, etc.)",21.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers,"critical thinking, the skill to define the tas...",60000.0,7.0


In [3]:
def log(step, before):
    """Print how many rows a step removed and how many remain."""
    print(f"{step}: removed {before - len(df):,} | remaining {len(df):,}")

In [4]:
mainbranch_filter = [
    "I am a developer by profession",
    "I am not primarily a developer, but I write code sometimes as part of my work/studies"
]
employment_filter = [
    "Employed",
    "Independent contractor, freelancer, or self-employed",
    "Student"
]

In [5]:
df = df_raw[
    df_raw["MainBranch"].isin(mainbranch_filter)
    & df_raw["Employment"].isin(employment_filter)
].copy()
print("after respondent filter:", df.shape)

after respondent filter: (39891, 172)


In [6]:
before = len(df)
df = df.drop_duplicates(subset=df.columns.drop("ResponseId")).copy()
log("duplicates", before)
 

duplicates: removed 649 | remaining 39,242


In [7]:
cols = [
    "Country",
    "Age",
    "EdLevel",
    "Employment",
    "MainBranch",
    "YearsCode",
    "DevType",
    "OrgSize",
    "ICorPM",
    "RemoteWork",
    "Industry",
    "LanguageHaveWorkedWith",
    "DatabaseHaveWorkedWith",
    "ConvertedCompYearly",
]
df = df[cols].copy()
print("Share of missing values per column (before dropping missing salaries):")
print(df.isna().mean().round(3))
 
before = len(df)
df = df[df["ConvertedCompYearly"].notna()].copy()
log("missing salary", before)

Share of missing values per column (before dropping missing salaries):
Country                   0.246
Age                       0.000
EdLevel                   0.002
Employment                0.000
MainBranch                0.000
YearsCode                 0.080
DevType                   0.072
OrgSize                   0.195
ICorPM                    0.215
RemoteWork                0.202
Industry                  0.206
LanguageHaveWorkedWith    0.314
DatabaseHaveWorkedWith    0.431
ConvertedCompYearly       0.442
dtype: float64
missing salary: removed 17,339 | remaining 21,903


In [8]:
## Remove salary outliers
# Rules (applied together):
# 1. Salary below 10% of the respondent's country median (likely currency/entry errors)
# 2. Salary above the 99th percentile
# 3. Salary below $1,000 (obvious mistakes, e.g. $1)
 
# %%
comp = df["ConvertedCompYearly"]
print(comp.describe())
print(comp.quantile([0.01, 0.05, 0.95, 0.99]))
 
country_median = df.groupby("Country")["ConvertedCompYearly"].transform("median")
high_cutoff = comp.quantile(0.99)
 
too_low_for_country = comp < 0.10 * country_median
too_high = comp > high_cutoff
below_floor = comp < 1000
 
print(f"\n< 10% of country median: {too_low_for_country.sum()}")
print(f"> 99th percentile (${high_cutoff:,.0f}): {too_high.sum()}")
print(f"< $1,000: {below_floor.sum()}")
print("Countries with the most 'too low' salaries:")
print(df.loc[too_low_for_country, "Country"].value_counts().head(10))
 
before = len(df)
df = df[~(too_low_for_country | too_high | below_floor)].copy()
log("salary outliers", before)
print(df["ConvertedCompYearly"].describe())

count    2.190300e+04
mean     9.934745e+04
std      2.964466e+05
min      1.000000e+00
25%      4.093500e+04
50%      7.652000e+04
75%      1.225270e+05
max      3.355272e+07
Name: ConvertedCompYearly, dtype: float64
0.01       102.04
0.05      3787.00
0.95    232029.00
0.99    420000.00
Name: ConvertedCompYearly, dtype: float64

< 10% of country median: 967
> 99th percentile ($420,000): 217
< $1,000: 503
Countries with the most 'too low' salaries:
Country
Ukraine                     134
United States of America     87
India                        55
Germany                      50
Brazil                       41
Poland                       36
Turkey                       24
Serbia                       19
Argentina                    18
Sweden                       18
Name: count, dtype: int64
salary outliers: removed 1,225 | remaining 20,678
count     20678.000000
mean      93106.395058
std       67306.393547
min        1000.000000
25%       46406.000000
50%       80000.000000
75% 

In [9]:
# ## Group rare countries
# Countries with fewer than 50 respondents are merged into "Other".
 
# %%
counts = df["Country"].value_counts()
rare = counts[counts < 50].index
df["Country"] = df["Country"].replace(rare, "Other")
 
print(f"{len(rare)} countries grouped into 'Other' ({(df['Country'] == 'Other').sum()} people)")
print("countries remaining:", df["Country"].nunique())

106 countries grouped into 'Other' (1187 people)
countries remaining: 51


In [ ]:
df = df.loc[(df['YearsCode'] <= 50)] # or df[["YearsCode"]] = df[["YearsCode"]].clip(upper=50)

In [13]:
# ## Handle missing values
# All "didn't answer" labels and NaNs in categorical columns become "Unknown".
# `YearsCode` stays numeric (NaN kept, to be imputed later if needed);
# `ConvertedCompYearly` has no missing values left.
 
# %%
unknown_labels = ["I don’t know", "Prefer not to say", "I prefer not to say"]  # curly ’ as in the survey
categorical_cols = [c for c in cols if c not in ("YearsCode", "ConvertedCompYearly")]
 
df[categorical_cols] = df[categorical_cols].replace(unknown_labels, "Unknown").fillna("Unknown")
 
print(df.isna().sum())

Country                   0
Age                       0
EdLevel                   0
Employment                0
MainBranch                0
YearsCode                 0
DevType                   0
OrgSize                   0
ICorPM                    0
RemoteWork                0
Industry                  0
LanguageHaveWorkedWith    0
DatabaseHaveWorkedWith    0
ConvertedCompYearly       0
dtype: int64


In [14]:
print("final shape:", df.shape)
df.head()

final shape: (20569, 14)


,Country,Age,EdLevel,Employment,MainBranch,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,Industry,LanguageHaveWorkedWith,DatabaseHaveWorkedWith,ConvertedCompYearly
0,Ukraine,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,I am a developer by profession,14.0,"Developer, mobile",20 to 99 employees,People manager,Remote,Fintech,Bash/Shell (all shells);Dart;SQL,Cloud Firestore;PostgreSQL,61256.0
1,Netherlands,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,I am a developer by profession,10.0,"Developer, back-end",500 to 999 employees,Individual contributor,"Hybrid (some in-person, leans heavy to flexibi...",Retail and Consumer Services,Java,Dynamodb;MongoDB,104413.0
2,Ukraine,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",I am a developer by profession,12.0,"Developer, front-end",Unknown,Unknown,Unknown,Software Development,Dart;HTML/CSS;JavaScript;TypeScript,MongoDB;MySQL;PostgreSQL,53061.0
3,Ukraine,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,I am a developer by profession,5.0,"Developer, back-end","10,000 or more employees",Individual contributor,Remote,Retail and Consumer Services,Java;Kotlin;SQL,Unknown,36197.0
4,Ukraine,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...",I am a developer by profession,22.0,Engineering manager,Unknown,Unknown,Unknown,Software Development,C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;P...,Elasticsearch;Microsoft SQL Server;MySQL;Oracl...,60000.0


In [ ]:
target_cols = [
    'LanguageChoice', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith',
    'LanguageAdmired', 'LanguagesHaveEntry', 'LanguagesWantEntry',
    'DatabaseChoice', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith',
    'DatabaseAdmired', 'DatabaseHaveEntry', 'DatabaseWantEntry',
    'PlatformChoice', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith',
    'PlatformAdmired', 'PlatformHaveEntry', 'PlatformWantEntry'
]


In [ ]:
print(df_raw[target_cols].notna().mean().sort_values())
def split(s): return set(s.split(';')) if isinstance(s, str) else set()
for grp, have, want, adm in [('Language','LanguageHaveWorkedWith','LanguageWantToWorkWith','LanguageAdmired'),
                             ('Database','DatabaseHaveWorkedWith','DatabaseWantToWorkWith','DatabaseAdmired'),
                             ('Platform','PlatformHaveWorkedWith','PlatformWantToWorkWith','PlatformAdmired')]:
    match = (df_raw[have].map(split) & df_raw[want].map(split)) == df_raw[adm].map(split)
    print(grp, "Admired == Have∩Want:", match.mean().round(3))


In [ ]:
df_raw[target_cols]

# 3. Create a clean, independent copy of the data
df_tech_stack = df_raw[target_cols].copy()
df_tech_stack.head()

In [ ]:
def multi_hot(df, cols, sep=';'):
    parts = []
    for c in cols:
        d = df[c].str.get_dummies(sep=sep)
        d.columns = [f"{c}__{x.strip()}" for x in d.columns]
        parts.append(d)
    return pd.concat(parts, axis=1)

use_cols = [c for c in target_cols if c in df_tech_stack.columns]
X = multi_hot(df_tech_stack, use_cols)

# Keep only respondents who answered at least one of these questions
X = X[df_tech_stack[use_cols].notna().any(axis=1)]

# Drop near-constant columns (they produce NaN or unstable correlations)
p = X.mean()
X = X.loc[:, (p > 0.005) & (p < 0.995)]

In [ ]:
keep = ['LanguageHaveWorkedWith', 'DatabaseHaveWorkedWith', 'PlatformHaveWorkedWith']

# 1. Keep only respondents who answered at least one of the three questions
d = df_tech_stack.loc[df_tech_stack[keep].notna().any(axis=1), keep]
print(d.shape)

# 2. Multi-hot encode
def multi_hot(d, cols, sep=';'):
    parts = []
    for c in cols:
        m = d[c].str.get_dummies(sep=sep)
        m.columns = [f"{c.replace('HaveWorkedWith','')}__{x.strip()}" for x in m.columns]
        parts.append(m)
    return pd.concat(parts, axis=1)

X = multi_hot(d, keep)

# 3. Drop rare items (<1% of respondents) so correlations are stable
p = X.mean()
X = X.loc[:, p >= 0.01]
print(X.shape)

# 4. Correlation matrix (phi coefficient for binary columns)
corr = X.corr()

# 5. Strongest pairs, excluding self-pairs
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().rename('phi').reset_index()
pairs.columns = ['a', 'b', 'phi']
pairs['cross_group'] = pairs.a.str.split('__').str[0] != pairs.b.str.split('__').str[0]

print("Top positive pairs:")
print(pairs.sort_values('phi', ascending=False).head(15))
print("\nTop negative pairs:")
print(pairs.sort_values('phi').head(15))
print("\nTop cross-group pairs (e.g. language vs database):")
print(pairs[pairs.cross_group].reindex(
    pairs[pairs.cross_group].phi.abs().sort_values(ascending=False).index).head(15))

# 6. Heatmap of the full matrix
plt.figure(figsize=(16, 14))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            xticklabels=False, yticklabels=False)
plt.title("Item correlation (Language / Database / Platform, HaveWorkedWith)")
plt.show()

# 7. Cross-group heatmap: languages vs databases
lang = [c for c in X if c.startswith('Language__')]
db = [c for c in X if c.startswith('Database__')]
plt.figure(figsize=(14, 10))
sns.heatmap(corr.loc[lang, db], cmap='coolwarm', center=0, vmin=-0.5, vmax=0.5)
plt.title("Languages vs Databases")
plt.show()

# 8. Drop highly correlated columns
def drop_correlated(corr, threshold=0.8):
    upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    return [c for c in upper.columns if (upper[c] > threshold).any()]

to_drop = drop_correlated(corr, threshold=0.8)
print("\nColumns to drop:", to_drop)
X_reduced = X.drop(columns=to_drop)
print(X_reduced.shape)

In [ ]:
print(df_raw[["WorkExp","YearsCode"]].corr())
import seaborn as sns
sns.heatmap(df_raw[["WorkExp","YearsCode","ConvertedCompYearly"]].corr(), annot=True, cmap='coolwarm')


In [ ]:
num = df_raw.select_dtypes('number').drop(columns=['ConvertedCompYearly'], errors='ignore')

def high_corr_pairs(corr, threshold=0.6):
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    s = upper.stack().rename('corr').reset_index()
    s.columns = ['a', 'b', 'corr']
    s = s[s['corr'].abs() >= threshold]
    return s.reindex(s['corr'].abs().sort_values(ascending=False).index)

print(high_corr_pairs(num.corr(method='spearman'), 0.6))

In [ ]:
df_raw['ICorPM'].value_counts(dropna=False).sort_index()

In [ ]:
dev_map = {
    # Software development
    "Developer, full-stack": "Full-stack",
    "Developer, back-end": "Back-end",
    "Developer, front-end": "Front-end",
    "Developer, mobile": "Mobile",
    "Developer, desktop or enterprise applications": "Desktop/Enterprise",
    "Developer, embedded applications or devices": "Embedded",
    "Developer, game or graphics": "Game/Graphics",
    "Architect, software or solutions": "Architecture",

    # Management and leadership
    "Engineering manager": "Management",
    "Senior executive (C-suite, VP, etc.)": "Management",
    "Founder, technology or otherwise": "Management",
    "Project manager": "Management",
    "Product manager": "Management",

    # Data, AI and analytics
    "Data engineer": "Data/AI/Analytics",
    "AI/ML engineer": "Data/AI/Analytics",
    "Data scientist": "Data/AI/Analytics",
    "Applied scientist": "Data/AI/Analytics",
    "Developer, AI apps or physical AI": "Data/AI/Analytics",
    "Data or business analyst": "Data/AI/Analytics",

    # Infrastructure and operations
    "DevOps engineer or professional": "Infrastructure/Operations",
    "Cloud infrastructure engineer": "Infrastructure/Operations",
    "System administrator": "Infrastructure/Operations",
    "Database administrator or engineer": "Infrastructure/Operations",

    # Security
    "Cybersecurity or InfoSec professional": "Security",

    # Quality assurance and support
    "Developer, QA or test": "QA/Testing",
    "Support engineer or analyst": "IT Support",

    # Academic and other
    "Academic researcher": "Academic/Research",
    "Student": "Student",
    "Retired": "Retired",
    "Other (please specify):": "Other",
    "Financial analyst or engineer": "Other",
    "UX, Research Ops or UI design professional": "Other",
}

df_raw["DevType_grp"] = df_raw["DevType"].map(dev_map).fillna("Other")

print(df_raw["DevType_grp"].value_counts())

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    r, k = ct.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1)
    kc = k - (k - 1) ** 2 / (n - 1)
    denom = min(kc - 1, rc - 1)
    return np.sqrt(phi2 / denom) if denom > 0 else np.nan
ct = pd.crosstab(df_raw['DevType_grp'], df_raw['ICorPM'], dropna=False)
print(ct)
print(pd.crosstab(df_raw['DevType_grp'], df_raw['ICorPM'], normalize='index').round(2))

print("Cramér's V:", cramers_v(df_raw['DevType_grp'], df_raw['ICorPM']))

In [ ]:
def high_corr_pairs(corr, threshold=0.6):
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    s = upper.stack().rename('corr').reset_index()
    s.columns = ['a', 'b', 'corr']
    s = s[s['corr'].abs() >= threshold]
    return s.reindex(s['corr'].abs().sort_values(ascending=False).index)

In [ ]:
t = 'ConvertedCompYearly'
print(df_raw.groupby('ICorPM', dropna=False)[t].agg(['count','median','mean']))
print(df_raw.groupby('DevType_grp', dropna=False)[t].agg(['count','median']))

In [ ]:
import statsmodels.formula.api as smf, numpy as np
d = df_raw[[t, 'ICorPM', 'DevType_grp']].dropna()
d['logy'] = np.log1p(d[t])

base = smf.ols('logy ~ C(DevType_grp)', d).fit()
full = smf.ols('logy ~ C(DevType_grp) + C(ICorPM)', d).fit()
print(full.compare_f_test(base))      # (F, p-value, df diff)
print(base.rsquared, full.rsquared)

In [ ]:
d = df_raw[[t, 'ICorPM', 'DevType_grp', 'Country', 'WorkExp']].dropna()
d['logy'] = np.log1p(d[t])

base = smf.ols('logy ~ C(DevType_grp) + C(Country) + WorkExp', d).fit()
full = smf.ols('logy ~ C(DevType_grp) + C(Country) + WorkExp + C(ICorPM)', d).fit()
print(full.compare_f_test(base))
print(base.rsquared, full.rsquared)